## Remember MNIST from S3 and Run Mock Training

### Objective
Recover the MNIST manifest from S3, remember the single archived dataset object from S3, decode it into tensors, and run a short mock PyTorch training loop.

In [ ]:
%load_ext autoreload
%autoreload 2

### Objective
Import the notebook dependencies, configure paths, and define the reusable constants for the example.

In [ ]:
import hashlib
import io
import json

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import laila
from laila.pool import S3Pool

POOL_NICKNAME = "mnist_s3_pool"
MANIFEST_NICKNAME = "mnist_s3_manifest"
BATCH_SIZE = 64
MAX_SAMPLES = 512
MAX_STEPS = 5

### Objective
Define a tiny MNIST network and the helper functions used to build the S3 pool, load the manifest, decode the recovered archive, and build a tensor dataset.

In [ ]:
class TinyMNISTNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )


def manifest_entry_id_from_nickname(manifest_nickname: str) -> str:
    # Nicknames deterministically map to entry IDs, so the manifest can be re-fetched from S3.
    return laila.constant(data=np.zeros(1, dtype=np.uint8), nickname=manifest_nickname).global_id


def entry_payload_to_bytes(payload: object) -> bytes:
    # The upload notebook stored the whole archive as one uint8 array.
    return np.asarray(payload, dtype=np.uint8).tobytes()


def load_manifest_from_s3(manifest_entry_id: str, pool_nickname: str) -> dict:
    manifest_future = laila.remember(manifest_entry_id, pool_nickname=pool_nickname)
    print("Manifest future type:", type(manifest_future).__name__)
    manifest_future.wait()
    manifest_entry = manifest_future.result
    return dict(manifest_entry.data)


def load_mnist_from_npz_bytes(npz_bytes: bytes, max_samples: int) -> tuple[torch.Tensor, torch.Tensor]:
    # Read the compressed MNIST archive directly from memory.
    with np.load(io.BytesIO(npz_bytes)) as data:
        images = data["x_train"][:max_samples].astype(np.float32) / 255.0
        labels = data["y_train"][:max_samples].astype(np.int64)

    image_tensor = torch.from_numpy(images[:, None, :, :])
    label_tensor = torch.from_numpy(labels)
    return image_tensor, label_tensor

### Objective
Register the S3 pool, derive the manifest entry ID from its deterministic nickname, and recover the manifest directly from S3.

In [ ]:
pool_nickname = POOL_NICKNAME
manifest_entry_id = manifest_entry_id_from_nickname(MANIFEST_NICKNAME)

s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=pool_nickname)

manifest = load_manifest_from_s3(manifest_entry_id, pool_nickname=pool_nickname)

print("Manifest entry id:", manifest_entry_id)
print("Archive entry id:", manifest["entry_id"])
print("Pool nickname:", pool_nickname)

### Objective
Remember the single MNIST archive object from S3, wait on the returned future, and verify that the reconstructed bytes still match the upload-time hash.

In [ ]:
# Single-object fetch: this returns a regular future rather than a GroupFuture.
remember_future = laila.remember(manifest["entry_id"], pool_nickname=pool_nickname)
print("Remember future type:", type(remember_future).__name__)

remember_future.wait()
recovered_entry = remember_future.result

archive_bytes = entry_payload_to_bytes(recovered_entry.data)
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
assert archive_sha256 == manifest["archive_sha256"]

print("Recovered archive size:", len(archive_bytes))
print("Recovered SHA256:", archive_sha256)

### Objective
Decode a small MNIST training subset from the recovered archive and inspect the resulting tensor shapes.

In [ ]:
images, labels = load_mnist_from_npz_bytes(archive_bytes, max_samples=MAX_SAMPLES)

print("Images shape:", tuple(images.shape))
print("Labels shape:", tuple(labels.shape))

### Objective
Create a `DataLoader`, build a tiny classifier, and run a short mock training loop to prove the recovered dataset can feed a normal PyTorch pipeline.

In [ ]:
loader = DataLoader(TensorDataset(images, labels), batch_size=BATCH_SIZE, shuffle=True)
model = TinyMNISTNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

model.train()
for step, (batch_images, batch_labels) in enumerate(loader, start=1):
    # Standard training step: forward, loss, backward, optimizer update.
    optimizer.zero_grad()
    logits = model(batch_images)
    loss = criterion(logits, batch_labels)
    loss.backward()
    optimizer.step()

    print(
        f"step={step:02d} "
        f"batch_shape={tuple(batch_images.shape)} "
        f"loss={loss.item():.4f}"
    )
    if step >= MAX_STEPS:
        break

print("Mock MNIST training complete.")